In [1]:
!python -V

Python 3.12.0


Cette version contient le deploiement sur MLFLOW

In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import os

In [3]:
a=os.curdir
os.listdir(a)

['delivery_forecasting.ipynb',
 'delivery_forecasting_v1.ipynb',
 'mlruns',
 'requirements.txt',
 'supply_chain_large.csv']

In [4]:
df=pd.read_csv("supply_chain_large.csv")

In [5]:
#5 premières lignes
df.head()
# #nombre de lignes du df
# len(df)
# #nombre d'element dans le df
# df.size
# #nombre de ligne et de colonnes
# df.shape

,order_id,product_id,origin_warehouse,destination_city,order_date,ship_mode,carrier,weight_kg,distance_km,expected_delay_days
0,100000,P052,Rouen,Bordeaux,2024-11-23,Standard,GLS,2.98,983,4
1,100001,P093,Nantes,Nice,2025-06-16,Express,Colissimo,12.82,724,3
2,100002,P015,Rouen,Marseille,2024-08-07,Standard,DHL,14.11,392,4
3,100003,P072,Rouen,Paris,2024-12-18,Express,Chronopost,12.38,380,2
4,100004,P061,Paris,Lyon,2025-03-30,Express,DHL,18.77,373,2


In [6]:
#vérifier s'il y a une valeur manquante
df.isnull().values.any()

np.False_

In [7]:
df.columns

Index(['order_id', 'product_id', 'origin_warehouse', 'destination_city',
       'order_date', 'ship_mode', 'carrier', 'weight_kg', 'distance_km',
       'expected_delay_days'],
      dtype='object')

In [8]:
df.dtypes

order_id                 int64
product_id              object
origin_warehouse        object
destination_city        object
order_date              object
ship_mode               object
carrier                 object
weight_kg              float64
distance_km              int64
expected_delay_days      int64
dtype: object

In [9]:
a= df.columns
a


Index(['order_id', 'product_id', 'origin_warehouse', 'destination_city',
       'order_date', 'ship_mode', 'carrier', 'weight_kg', 'distance_km',
       'expected_delay_days'],
      dtype='object')

In [10]:
def convert_to_string(df):
    a= df.columns
    for i in list(a):
        print(i)
        # df[f"i"]=df[f"i"].astype(str)
        return type(a)

In [11]:
convert_to_string(df)

order_id


pandas.core.indexes.base.Index

In [12]:
df["product_id"] = df["product_id"].astype(str)
df["product_id"] = df["product_id"].astype(str)


In [13]:
df.dtypes
df.columns

Index(['order_id', 'product_id', 'origin_warehouse', 'destination_city',
       'order_date', 'ship_mode', 'carrier', 'weight_kg', 'distance_km',
       'expected_delay_days'],
      dtype='object')

In [14]:
#séparer le jeu de données en var categorielles et numériques
# categorical=["origin_warehouse","ship_mode","ship_mode"]
categorical=["product_id","destination_city","carrier"]
numerical=["weight_kg"]

In [15]:
#convertir les variables categorielles en str pour faciliter la vectorization
df[categorical]=df[categorical].astype(str)

In [ ]:
#concatene les 2 listes categorical et numerical en dictionnaire
train_dicts=df[categorical + numerical].to_dict(orient='records')
train_dicts
#si on veut travailler sur un dictionnaire

In [17]:
a=df[categorical]
b=df[numerical]

# df1=pd.concat([a, b], ignore_index=True)
# df1


In [18]:
encoder = OneHotEncoder(sparse_output=False)  # sparse_output=False pour un tableau dense (lisible)

# Appliquer l'encodage
cat=encoder.fit_transform(df[categorical])
# encoder.get_feature_names_out()
type(cat)
cat

array([[0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 1., 0., 0.]])

In [ ]:
#variable explicatives
features = np.hstack((cat, df["weight_kg"].values.reshape(-1, 1)))
len(features)
#variable cible
target=df["expected_delay_days"]
target

In [20]:
#separation de jeu de données
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)


Sur le terminal executer:
mlflow ui

In [25]:
%mlflow ui

UsageError: Line magic function `%mlflow` not found.


In [ ]:
with mlflow.start_run():
    models = {
        "Linear": LinearRegression(),
        "Ridge": Ridge(),
        "Lasso": Lasso(),
        "RandomForest": RandomForestRegressor(),
        "GradientBoosting": GradientBoostingRegressor()
    }
    
    # Prédictions et évaluation
if mlflow.active_run() is not None:
    mlflow.end_run()
for name, model in models.items():
        with mlflow.start_run(run_name=name):  # ✅ nouveau run à chaque modèle
            mlflow.set_tag("developer","Anas")
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            r2 = r2_score(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)

            # Log dans MLflow
            mlflow.log_param("model_type", name)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("mse", mse)
            mlflow.sklearn.log_model(model, "model")

            print(f"\n{name} loggé dans MLflow → R²: {r2:.3f}, MAE: {mae:.3f}, MSE: {mse:.3f}")